#solve the txt to json,only R1 need it

In [1]:
import re
import json
from pathlib import Path

# 路径按你的项目结构来：notebook 在 jupyter/，txt 在 data/
txt_path = Path("../data/20260330_cn.txt")
out_path = Path("../src/sft_qwen3_14b_out/R2_candidates.json")

text = txt_path.read_text(encoding="utf-8")

# 按 "## 1．R2-01 ..." 这种结构拆块
blocks = re.split(r"^##\s+\d+．", text, flags=re.M)
headers = re.findall(r"^##\s+\d+．\s*(.+)$", text, flags=re.M)

candidates = []
TOTAL_MASS = 20.0  # “材料组成，20.00 g”

for header, block in zip(headers, blocks[1:]):
    # 从标题里抓 R2-01 / R2-02 ...
    m_id = re.search(r"(R2-\d+)", header)
    if not m_id:
        continue
    cid = m_id.group(1)

    # ---------- 材料组成 ----------
    m_mat = re.search(r"###\s*材料组成[^\n]*\n(.*?)(?:\n###|\Z)", block, flags=re.S)
    materials = []
    pva_mass = 0.0

    if m_mat:
        for line in m_mat.group(1).splitlines():
            line = line.strip()
            if not line.startswith("*"):
                continue
            # 形如 "* 聚乙烯醇 PVA：2.40 g"
            name_part = line[1:].split("：", 1)[0].strip()
            mass_m = re.search(r"([\d.]+)\s*g", line)
            mass = float(mass_m.group(1)) if mass_m else None
            materials.append({"name_raw": name_part, "mass_g": mass})
            if "PVA" in name_part or "聚乙烯醇" in name_part:
                if mass is not None:
                    pva_mass = mass

    pva_wt_percent = round(pva_mass / TOTAL_MASS * 100, 2) if pva_mass else None

    # PVA、水 视为基体，其它算作添加剂
    additives = []
    for m in materials:
        name = m["name_raw"]
        if any(k in name for k in ["PVA", "聚乙烯醇", "去离子水", "DI水", "去离子 水"]):
            continue
        wt_percent = round(m["mass_g"] / TOTAL_MASS * 100, 2) if m["mass_g"] else None
        additives.append({
            "name": name,
            "wt_percent": wt_percent,
        })

    # ---------- 制备流程 ----------
    m_proc = re.search(r"###\s*制备流程\s*\n(.*?)(?:\n###|\Z)", block, flags=re.S)
    steps = []
    if m_proc:
        for line in m_proc.group(1).splitlines():
            line = line.strip()
            if not line:
                continue
            # 去掉前面的 "1."、"2." 等编号
            line = re.sub(r"^\d+\.\s*", "", line)
            steps.append(line)

    # ---------- 配方用途 ----------
    m_use = re.search(r"###\s*配方用途\s*\n(.*?)(?:\n---|\Z)", block, flags=re.S)
    use_text = ""
    if m_use:
        use_text = " ".join(l.strip() for l in m_use.group(1).splitlines() if l.strip())

    # 简单根据是否包含“光固化”来区分网络类型
    network_type = "photo" if ("光固化" in header or "光固化" in block) else "chemical"
    crosslink = "uv_photo" if network_type == "photo" else "ga_hcl_fast"

    candidate = {
        "candidate_id": cid,
        "formulation": {
            "pva_wt_percent": pva_wt_percent,
            "additives": additives,
            "network_type": network_type,
            "crosslink_or_phys_method": crosslink,
        },
        "processing": {
            "steps": steps,
            # 这些字段只是给 DOE 用的元数据，不影响流程，先给一个合理占位
            "freeze_thaw_cycles": 0,
            "freeze_temp_C": None,
            "thaw_temp_C": None,
            "cycle_hours": None,
            "post_soak_hours": 1,
        },
        "expected_mechanism": [],
        "risks_and_mitigations": [],
        "predicted_tradeoff": {
            "cof_trend": "unknown",
            "wear_trend": "unknown",
            "stability_trend": "unknown",
            "notes": use_text,
        },
        "confidence": 0.6,
    }
    candidates.append(candidate)

obj = {
    # 与现有 R1_candidates.json 中的 constraints 一致
    "constraints": {
        "load_N": 10.0,
        "counterface": "stainless_steel_ball",
        "medium": "DI_water",
        "temperature": "25℃_room_temp",
        "speed_mm_s": 1.0,
    },
    "candidates": candidates,
}

out_path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {len(candidates)} candidates to {out_path}")

Wrote 0 candidates to ../src/sft_qwen3_14b_out/R2_candidates.json


convert csv to request csv

In [5]:
import csv
import re
from pathlib import Path

src_csv = Path("../data/20260330.csv")
dst_csv = Path("../src/sft_qwen3_14b_out/R2_results_template.csv")

def convert_results(src: Path, dst: Path):
    fieldnames = [
        "candidate_id",
        "cof_steady_mean",
        "cof_std",
        "wear_proxy",
        "compression_modulus_MPa",
        "failure_type",
        "failure_time_min",
        "notes",
        # 额外保留三组原始 COF
        "COF_mean_1",
        "COF_std_1",
        "COF_mean_2",
        "COF_std_2",
        "COF_mean_3",
        "COF_std_3",
    ]

    # 仍然用 utf-8-sig 去掉 BOM
    with src.open(encoding="utf-8-sig") as f_in, dst.open("w", encoding="utf-8", newline="") as f_out:
        reader = csv.DictReader(f_in)
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            cid_raw = (row.get("R") or "").strip()
            if not cid_raw:
                continue

            # 标准化成 R2-01 这种两位数格式
            m = re.match(r"R2-(\d+)", cid_raw)
            if m:
                idx = int(m.group(1))
                cid = f"R2-{idx:02d}"
            else:
                cid = cid_raw

            cof1 = (row.get("COF_mean_1") or "").strip()
            std1 = (row.get("COF_std_1") or "").strip()
            cof2 = (row.get("COF_mean_2") or "").strip()
            std2 = (row.get("COF_std_2") or "").strip()
            cof3 = (row.get("COF_mean_3") or "").strip()
            std3 = (row.get("COF_std_3") or "").strip()
            modulus = (row.get("Modulus_MPa") or "").strip()

            failure_type = "none"
            notes = ""

            if cof3.upper().startswith("ERROR1"):
                # 水凝胶撑不住 10 N，没法测摩擦
                failure_type = "cant_hold_10N"
                notes = "ERROR1: gel could not sustain 10 N normal load; test aborted."
                cof_out = ""
                std_out = ""
                modulus_out = ""
            elif cof3.upper().startswith("ERROR2"):
                # 摩擦过程中破损：COF 没有可信值，但压缩模量如果先测了就保留
                failure_type = "gel_breakage"
                notes = "ERROR2: gel damaged / broken during friction test."
                cof_out = ""
                std_out = ""
                modulus_out = modulus
            elif cof3.upper().startswith("ERROR3"):
                # 未成胶
                failure_type = "not_gelled"
                notes = "ERROR3: gel did not form."
                cof_out = ""
                std_out = ""
                modulus_out = ""
            else:
                # 正常数值样本：仍然用第 3 组作为 steady 值
                cof_out = cof3
                std_out = std3
                modulus_out = modulus

            writer.writerow({
                "candidate_id": cid,
                "cof_steady_mean": cof_out,
                "cof_std": std_out,
                "wear_proxy": "",
                "compression_modulus_MPa": modulus_out,
                "failure_type": failure_type,
                "failure_time_min": "",
                "notes": notes,
                # 三组原始 COF 全部保留
                "COF_mean_1": cof1,
                "COF_std_1": std1,
                "COF_mean_2": cof2,
                "COF_std_2": std2,
                "COF_mean_3": cof3,
                "COF_std_3": std3,
            })

convert_results(src_csv, dst_csv)
print(f"Converted results to: {dst_csv}")

Converted results to: ../src/sft_qwen3_14b_out/R2_results_template.csv
